In [ ]:
#--- Macroeconomics Features Only ---
# Using the late fusion syntax with a few adjustment

#--- Import necessary packages ---
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL 
import numpy as np
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import InputLayer, LSTM, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import IPython.display as display
import os
import random
import tensorflow as tf
import copy
from sklearn.metrics import mean_absolute_error, mean_squared_error
import tensorflow.keras.backend as K
# -----------------------------------------------------------------------------------------------------------------------------------

# -----------------------------------------------------------------------------------------------------------------------------------
#--- Customized loss function ---
def customTrendLoss(lambda_, delta, penalty_factor, threshold):
    def loss(y_true, y_pred):
        # The formula of Basic Error
        error = y_true - y_pred
        abs_error = K.abs(error)

        # The formula of Huber Loss
        is_small = K.less_equal(abs_error, delta)
        huber = tf.where(is_small, 0.5 * K.square(error), delta * (abs_error - 0.5 * delta))
        huber_loss = K.mean(huber)

        # The formula of Directional Penalty
        def thresholdedSign(x):
            return tf.where(x < -threshold, -1.0, tf.where(x > threshold, 1.0, 0.0))

        directionTrue = thresholdedSign(y_true)
        directionPred = thresholdedSign(y_pred)
        directionMismatch = K.cast(K.not_equal(directionTrue, directionPred), K.floatx())
        direction_penalty = K.mean(directionMismatch * penalty_factor * abs_error)

        # The formula of combined custom components
        custom = huber_loss + direction_penalty

        # The formula of MSE for stability
        mseLoss = K.mean(K.square(error))

        finalLoss = lambda_ * mseLoss + (1.0 - lambda_) * custom
        return finalLoss
    
    return loss
# -----------------------------------------------------------------------------------------------------------------------------------

# -----------------------------------------------------------------------------------------------------------------------------------
#--- Makes the model's output stable ---

np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)
# -----------------------------------------------------------------------------------------------------------------------------------

# -----------------------------------------------------------------------------------------------------------------------------------
#--- Defines all significant variables ---

# Defines the timesteps variable
timesteps = 8
# Defines the interval variable
interval = 7
# Defines the output Timesteps variable
outputTimesteps = 8
# -----------------------------------------------------------------------------------------------------------------------------------

# -----------------------------------------------------------------------------------------------------------------------------------
#--- Defines all function for the evaluation metrics (MAPE, SMAPE, and trend accuracy) ---

# Defines function for the MAPE metric formula (%)
def MAPE(y_true, y_prediction):
    epsilon = 1e-10  
    return np.mean(np.abs((y_true - y_prediction) / (y_true + epsilon))) * 100

# Defines function for the SMAPE metric formula (%)
def SMAPE(y_true, y_pred):
    epsilon = 1e-10  
    return 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + epsilon))

# Defines function for the trend accuracy
def trendAccuracy(y_actualBefore, y_actual, y_predictedBefore, y_predicted, threshold):
  
    trendActual = ((y_actual-y_actualBefore)/y_actualBefore) * 100
    trendPredicted = ((y_predicted-y_predictedBefore)/y_predictedBefore) * 100

    def labelTrend(trendValues, threshold):
        labels = np.zeros_like(trendValues, dtype=int)
        labels[trendValues > threshold] = 1 
        labels[trendValues < -threshold] = -1
        return labels
    
    actual_labels = labelTrend(trendActual, threshold)
    predicted_labels = labelTrend(trendPredicted, threshold)

    correct_predictions = np.sum(actual_labels == predicted_labels)
    accuracy = correct_predictions / len(actual_labels)

    return accuracy
# -----------------------------------------------------------------------------------------------------------------------------------

# -----------------------------------------------------------------------------------------------------------------------------------
#--- Preprocesses the target or y variable (Daily WTI Crude Oil Spot Price) ---

# Loads the Daily WTI Crude Oil Spot Price from a .csv file (from 2015 until 2019) (a period of 5 years) 
priceData = pd.read_csv('Crude Oil WTI Spot Price (5 Years) - copy.csv') 
# print(priceData.shape) # Shape [1254, 2]

# Transforms the variable in the 'Date' column into executable variable (for filling the missing values in the next step)
priceData['Date'] = pd.to_datetime(priceData['Date'], format='%d-%b-%y')
# Transforms the 'Date' column into an index of the dataframe
priceData.set_index('Date', inplace=True)
# print(priceData.shape) # Shape [1254, 1]

# Fills and handles the missing values (missing values including holidays in weekdays and weekends)
allDates = pd.date_range(start=priceData.index.min(), end=priceData.index.max(), freq='D') # freq = 'D' means all avilable days between the start and end date
priceData = priceData.reindex(allDates)
priceData.index.name = 'Date'
# Fills missing values using interpolation method
priceData['Price'] = priceData['Price'].interpolate(method='linear').round(2) 
# print(priceData.shape) # Shape [1825, 1] # The day of the date starts from Monday and ends on Friday (1825 days + 2 days = 1827 / 7 = 261 weeks in total)
# Removes the last 5 rows because of the incompleteness (no saturday and sunday on that week) and exceeding the year of 2019
priceData = priceData.iloc[:-5, :]
# print(priceData.shape) # Shape [1820, 1]

# Defines a function to visualize the daily crude oil spot prices (original and decomposition components)
# This function is for visualization purpose only
def visualizePriceDecomposition(priceData):
    plt.figure(figsize=(12, 8))
    # All components (original daily crude oil spot price) (additive methodology)
    plt.subplot(4, 1, 1)
    plt.plot(priceData['Price'], label='WTI Crude Oil Spot Price')
    plt.title('Original Prices', fontsize=10)
    plt.legend()
    # Trend component
    plt.subplot(4, 1, 2)
    plt.plot(priceData['trend'], label='Trend')
    plt.title('Trend Component', fontsize=10)
    plt.legend()
    # Seasonal component
    plt.subplot(4, 1, 3)
    plt.plot(priceData['seasonal'], label='Seasonality')
    plt.title('Seasonal Component', fontsize=10)
    plt.legend()
    # Residuals component
    plt.subplot(4, 1, 4)
    plt.plot(priceData['residual'], label='Residuals')
    plt.title('Residual Component', fontsize=10)
    plt.legend()
    plt.tight_layout()
    plt.show()

# Calls and implements the decomposition package (decomposing into trend, seasonal, and residual components) and rounds these components into two decimals place after
stl = STL(priceData['Price'], period=28) # 28 represents the number of days in a month 
result = stl.fit()
priceData['trend'] = result.trend.round(2)
priceData['seasonal'] = result.seasonal.round(2)
priceData['residual'] = result.resid.round(2)
# print(priceData.shape) # Shape [1820, 4]
# Calls the visualization function
# visualizePriceDecomposition(priceData)

# Adds the 'price_diff', 'trend_diff', 'seasonal_diff', 'residual_diff' columns because the research wants to predict the diff rather than the price
# The _diff uses a weekly shift (7 days shifting)
# Eventhough the dataset is daily price, this research wants to make it to have a weekly behavior
priceData['price_diff'] = priceData['Price'] - priceData['Price'].shift(7)
priceData['trend_diff'] = priceData['trend'] - priceData['trend'].shift(7)
priceData['seasonal_diff'] = priceData['seasonal'] - priceData['seasonal'].shift(7)
priceData['residual_diff'] = priceData['residual'] - priceData['residual'].shift(7)
# print(priceData.shape) # Shape [1820, 8]
# Removes the first 7 rows because of the NaN values on the '_diff' columns
priceData = priceData.iloc[7:]
# print(priceData.shape) # Shape [1813, 8]

# Note: the variables below will not be used on the fusion strategies
# Adds a column with the day of week status (Monday, Tuesday, etc.)
priceData['DayOfWeek'] = priceData.index.day_name()
daysOrder = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
priceData = pd.get_dummies(priceData, columns=['DayOfWeek'])
# Renames columns to remove the 'DayOfWeek_' prefix
renameMapping = {f'DayOfWeek_{day}': day for day in daysOrder}
priceData.rename(columns=renameMapping, inplace=True)
# Converts boolean 'True/False' values to integer '1/0' (one-hot enconding) for simplicity
priceData[daysOrder] = priceData[daysOrder].astype(int)
# Reorders the columns order on the dataframe 
priceData = priceData[['Price', 'trend', 'seasonal', 'residual', 'price_diff', 'trend_diff', 'seasonal_diff', 'residual_diff'] + daysOrder]
# print(priceData.shape) # Shape [1813, 15]
# -----------------------------------------------------------------------------------------------------------------------------------

# -----------------------------------------------------------------------------------------------------------------------------------
#--- Preprocess the x variables (macroeconomics features) ---

# Loads a csv file that contains the list of the choosen macroeconomics variables from granger casuality test
selectedMacroVariables = pd.read_csv('CausingVariables.csv', header=None) 
# Converts the dataframe into a list, removes the first variable on the list (the header of the list) and adds a 'DATE' column into the list
selectedMacroVariables = selectedMacroVariables[0].tolist()
selectedMacroVariables = selectedMacroVariables[1:]
selectedMacroVariables.insert(0, 'DATE')
# print(len(selectedMacroVariables)) # Shape [431]

# Loads a csv file containing the macroeconomics variables dataset (differencing once) (already stationary and non-constant)
macroeconomicsData = pd.read_csv("MacroeconomicsDailyFilteredStationaryCleaned.csv") 
# print(macroeconomicsData.shape) # Shape [1813, 8867]

# Chooses the macroeconomics variables that only contained in the list (filtering the macroeconomics dataset)
finalMacroeconomicData = macroeconomicsData.copy()
finalMacroeconomicData = finalMacroeconomicData[selectedMacroVariables]
# print(finalMacroeconomicData.shape) # Shape [1813, 431]
# Transforms the 'DATE' from columns to index of the dataframe
finalMacroeconomicData.set_index("DATE", inplace=True) 
# print(finalMacroeconomicData.shape) # Shape [1813, 430]
# -----------------------------------------------------------------------------------------------------------------------------------

# -----------------------------------------------------------------------------------------------------------------------------------
#--- Preprocess the x variables (textual features) (FinBERT Embeddings) (can be original or GPT-generated features) ---

# Loads a .csv file containing weekly sentence embedding within the called feautures
sentenceEmbeddingsData = pd.read_csv('Original Headline Sentence Embedding FinBERTku.csv')
# print(sentenceEmbeddingsData.shape) # Shape [1826, 770] # 5 years between 2015 and 2019 contains 1826 days

# Removes the first 11 rows and last 2 rows aligning with the price dataset
sentenceEmbeddingsData = sentenceEmbeddingsData.iloc[11: , : ]
sentenceEmbeddingsData = sentenceEmbeddingsData.iloc[ :-2 , : ]
# print(sentenceEmbeddingsData.shape) # Shape [1813, 770]

# Ensures the 'Date' column is in the correct and executable format
# sentenceEmbeddingsData['Date'] = pd.to_datetime(sentenceEmbeddingsData['Date'])
# Transforms the 'Date' from column to index of the dataframe 
sentenceEmbeddingsData.set_index('Date', inplace=True)
# print(sentenceEmbeddingsData.shape) # Shape [1813, 769]
# Deletes the first column of the dataframe (the raw textual feature)
sentenceEmbeddingsData=sentenceEmbeddingsData.iloc[ : , 1: ] 
# print(sentenceEmbeddingsData.shape) # Shape [1813, 768]
# -----------------------------------------------------------------------------------------------------------------------------------

# -----------------------------------------------------------------------------------------------------------------------------------
#--- Preprocesses all the X variables and y variable so it can be putted on the RNN models (train, validation, and test) phases ---

# Defines the Standard Scaler independently for each y variable (the one hot encoding variables are not scalled)
priceScaler = StandardScaler()
trendScaler = StandardScaler()
seasonalScaler = StandardScaler()
residualScaler = StandardScaler()
price_diffScaler = StandardScaler()
trend_diffScaler = StandardScaler()
seasonal_diffScaler = StandardScaler()
residual_diffScaler = StandardScaler()
# Defines the Standard Scaler for the X variables (one scaller for all the macroeconomics variables)
macroeconomicScaler = StandardScaler()

# Scales each column with defined variables (independently)
priceData.loc[:, 'Price'] = priceScaler.fit_transform(priceData[['Price']])
priceData.loc[:, 'trend'] = trendScaler.fit_transform(priceData[['trend']])
priceData.loc[:, 'seasonal'] = seasonalScaler.fit_transform(priceData[['seasonal']])
priceData.loc[:, 'residual'] = residualScaler.fit_transform(priceData[['residual']])
priceData.loc[:, 'price_diff'] = price_diffScaler.fit_transform(priceData[['price_diff']])
priceData.loc[:, 'trend_diff'] = trend_diffScaler.fit_transform(priceData[['trend_diff']])
priceData.loc[:, 'seasonal_diff'] = seasonal_diffScaler.fit_transform(priceData[['seasonal_diff']])
priceData.loc[:, 'residual_diff'] = residual_diffScaler.fit_transform(priceData[['residual_diff']])
# Scales all columns with a defined variable
finalMacroeconomicData[:] = macroeconomicScaler.fit_transform(finalMacroeconomicData)

# Defines the sequence functions (trend numerical, trend textual, seasonal, residuals numerical, residuals textual) with an interval value (7 days interval)
# Uses previous 8 weeks sequence to predict the next 8 weeks sequence
# Defines the trend numerical sequence function
def createTrendNumericalSequence(pricedf, macroeconomicsdf, timesteps, outputTimesteps, interval, targetColumn, priceColumn):
    X, y, price, priceBefore = [], [], [], []
    choosenPriceColumns = [priceColumn, targetColumn]
    
    for i in range(len(pricedf) - (timesteps+outputTimesteps) * interval):
        Xprice = pricedf[choosenPriceColumns].iloc[i:i + timesteps * interval: interval].values
        Xmacroeconomics = macroeconomicsdf.iloc[i:i + timesteps * interval: interval].values
    
        X.append(Xmacroeconomics)
       
        y.append(pricedf[[targetColumn]].iloc[i+timesteps*interval: i+(timesteps+outputTimesteps)*interval: interval].values.flatten())
        price.append(pricedf[[priceColumn]].iloc[i+timesteps*interval: i+(timesteps+outputTimesteps)*interval: interval].values.flatten())
        priceBefore.append(pricedf[[priceColumn]].iloc[i+timesteps*interval-interval].values.flatten())

    return np.array(X), np.array(y), np.array(price), np.array(priceBefore)

# Defines the trend textual sequence function
def createTrendTextualSequence(pricedf, sentenceEmbeddingsdf, timesteps, outputTimesteps, interval, targetColumn, priceColumn):
    X, y, price, priceBefore = [], [], [], []
    
    for i in range(len(pricedf) - (timesteps+outputTimesteps) * interval):
        XsentenceEmbeddings = sentenceEmbeddingsdf.iloc[i:i + timesteps * interval: interval].values
    
        X.append(XsentenceEmbeddings)
       
        y.append(pricedf[[targetColumn]].iloc[i+timesteps*interval: i+(timesteps+outputTimesteps)*interval: interval].values.flatten())
        price.append(pricedf[[priceColumn]].iloc[i+timesteps*interval: i+(timesteps+outputTimesteps)*interval: interval].values.flatten())
        priceBefore.append(pricedf[[priceColumn]].iloc[i+timesteps*interval-interval].values.flatten())

    return np.array(X), np.array(y), np.array(price), np.array(priceBefore)

# Defines the seasonal sequence functions (price as the target variable)
def createPriceSequence(pricedf, timesteps, outputTimesteps, interval, targetColumn, priceColumn):
    X, y, price, priceBefore = [], [], [], []
    choosenPriceColumns = [priceColumn, targetColumn]
    
    for i in range(len(pricedf) - (timesteps+outputTimesteps) * interval):
        Xprice = pricedf[choosenPriceColumns].iloc[i:i + timesteps * interval: interval].values
        
        X.append(Xprice)
        
        y.append(pricedf[[priceColumn]].iloc[i+timesteps*interval: i+(timesteps+outputTimesteps)*interval: interval].values.flatten())
        price.append(pricedf[[priceColumn]].iloc[i+timesteps*interval: i+(timesteps+outputTimesteps)*interval: interval].values.flatten())
        priceBefore.append(pricedf[[priceColumn]].iloc[i+timesteps*interval-interval].values.flatten())

    return np.array(X), np.array(y), np.array(price), np.array(priceBefore)

# Defines the numerical residuals sequence functions (price as the target variable)
def createPriceResidualsNumericalSequence(pricedf, timesteps, outputTimesteps, interval, targetColumn, priceColumn):
    X, y, price, priceBefore = [], [], [], []
    choosenPriceColumns = [priceColumn, targetColumn]
    
    for i in range(len(pricedf) - (timesteps+outputTimesteps) * interval):
        Xprice = pricedf[choosenPriceColumns].iloc[i:i + timesteps * interval: interval].values
        
        X.append(Xprice)
        
        y.append(pricedf[[priceColumn]].iloc[i+timesteps*interval: i+(timesteps+outputTimesteps)*interval: interval].values.flatten())
        price.append(pricedf[[priceColumn]].iloc[i+timesteps*interval: i+(timesteps+outputTimesteps)*interval: interval].values.flatten())
        priceBefore.append(pricedf[[priceColumn]].iloc[i+timesteps*interval-interval].values.flatten())

    return np.array(X), np.array(y), np.array(price), np.array(priceBefore)

# Defines the textual residuals sequence functions (price as the target variable)
def createPriceResidualsTextualSequence(pricedf, sentenceEmbeddingsdf, timesteps, outputTimesteps, interval, targetColumn, priceColumn):
    X, y, price, priceBefore = [], [], [], []
    
    for i in range(len(pricedf) - (timesteps+outputTimesteps) * interval):
        XsentenceEmbeddings = sentenceEmbeddingsdf.iloc[i:i + timesteps * interval: interval].values
        
        X.append(XsentenceEmbeddings)
        
        y.append(pricedf[[priceColumn]].iloc[i+timesteps*interval: i+(timesteps+outputTimesteps)*interval: interval].values.flatten())
        price.append(pricedf[[priceColumn]].iloc[i+timesteps*interval: i+(timesteps+outputTimesteps)*interval: interval].values.flatten())
        priceBefore.append(pricedf[[priceColumn]].iloc[i+timesteps*interval-interval].values.flatten())

    return np.array(X), np.array(y), np.array(price), np.array(priceBefore)

# Calls the function and creates sequences for the train, validation, and test datasets
X_trendNumerical, y_trendNumerical, y_priceTrendNumerical, y_priceBeforeTrendNumerical = createTrendNumericalSequence(priceData, finalMacroeconomicData, timesteps, outputTimesteps, interval, 'trend_diff', 'trend')
X_trendTextual, y_trendTextual, y_priceTrendTextual, y_priceBeforeTrendTextual = createTrendTextualSequence(priceData, sentenceEmbeddingsData, timesteps, outputTimesteps, interval, 'trend_diff', 'trend')
X_seasonal, y_seasonal, y_priceSeasonal, y_priceBeforeSeasonal = createPriceSequence(priceData, timesteps, outputTimesteps, interval, 'seasonal_diff', 'seasonal')
X_residualNumerical, y_residualNumerical, y_priceResidualNumerical, y_priceBeforeResidualNumerical = createPriceResidualsNumericalSequence(priceData, timesteps, outputTimesteps, interval, 'residual_diff', 'residual')
X_residualTextual, y_residualTextual, y_priceResidualTextual, y_priceBeforeResidualTextual = createPriceResidualsTextualSequence(priceData, sentenceEmbeddingsData, timesteps, outputTimesteps, interval, 'residual_diff', 'residual')

# # Prints the shape of each X sequence and y sequence
# print("==== Shape Summary ====")
# print(">> Trend Component (Numerical):")
# print(f"X_trendNumerical shape: {X_trendNumerical.shape}") # Shape [1701, 8, 432]
# print(f"y_trendNumerical shape: {y_trendNumerical.shape}") # Shape [1701, 8] 
# print(f"y_priceTrendNumerical shape: {y_priceTrendNumerical.shape}") # Shape [1701, 8] 
# print(f"y_priceBeforeTrendNumerical shape: {y_priceBeforeTrendNumerical.shape}") # Shape [1701, 1] 

# print("\n>> Trend Component (Textual):")
# print(f"X_trendTextual shape: {X_trendTextual.shape}") # Shape [1701, 8, 768]
# print(f"y_trendTextual shape: {y_trendTextual.shape}") # Shape [1701, 8] 
# print(f"y_priceTrendTextual shape: {y_priceTrendTextual.shape}") # Shape [1701, 8] 
# print(f"y_priceBeforeTrendTextual shape: {y_priceBeforeTrendTextual.shape}") # Shape [1701, 1] 

# print("\n>> Seasonal Component:")
# print(f"X_seasonal shape: {X_seasonal.shape}") # Shape [1701, 8, 2]
# print(f"y_seasonal shape: {y_seasonal.shape}") # Shape [1701, 8] 
# print(f"y_priceSeasonal shape: {y_priceSeasonal.shape}") # Shape [1701, 8] 
# print(f"y_priceBeforeSeasonal shape: {y_priceBeforeSeasonal.shape}") # Shape [1701, 1] 

# print("\n>> Residual Component (Numerical):")
# print(f"X_residualNumerical shape: {X_residualNumerical.shape}")  # Shape [1701, 8, 2]
# print(f"y_residualNumerical shape: {y_residualNumerical.shape}")  # Shape [1701, 8]
# print(f"y_priceResidualNumerical shape: {y_priceResidualNumerical.shape}")  # Shape [1701, 8]
# print(f"y_priceBeforeResidualNumerical shape: {y_priceBeforeResidualNumerical.shape}")  # Shape [1701, 1]

# print("\n>> Residual Component (Textual):")
# print(f"X_residualTextual shape: {X_residualTextual.shape}")  # Shape [1701, 8, 768]
# print(f"y_residualTextual shape: {y_residualTextual.shape}")  # Shape [1701, 8]
# print(f"y_priceResidualTextual shape: {y_priceResidualTextual.shape}")  # Shape [1701, 8]
# print(f"y_priceBeforeResidualTextual shape: {y_priceBeforeResidualTextual.shape}")  # Shape [1701, 1]
# print("========================")

# Defines lists for train, validation, and test datasets for each component (3 folds cross-validation)
X_trendNumericalTrains, y_trendNumericalTrains, y_priceTrendNumericalTrains, X_trendNumericalValidations, y_trendNumericalValidations, y_priceTrendNumericalValidations, X_trendNumericalTests, y_trendNumericalTests, y_priceTrendNumericalTests = [], [], [], [], [], [], [], [], []
X_trendTextualTrains, y_trendTextualTrains, y_priceTrendTextualTrains, X_trendTextualValidations, y_trendTextualValidations, y_priceTrendTextualValidations, X_trendTextualTests, y_trendTextualTests, y_priceTrendTextualTests = [], [], [], [], [], [], [], [], []
X_seasonalTrains, y_seasonalTrains, y_priceSeasonalTrains, X_seasonalValidations, y_seasonalValidations, y_priceSeasonalValidations, X_seasonalTests, y_seasonalTests, y_priceSeasonalTests = [], [], [], [], [], [], [], [], []
X_residualNumericalTrains, y_residualNumericalTrains, y_priceResidualNumericalTrains, X_residualNumericalValidations, y_residualNumericalValidations, y_priceResidualNumericalValidations, X_residualNumericalTests, y_residualNumericalTests, y_priceResidualNumericalTests = [], [], [], [], [], [], [], [], []
X_residualTextualTrains, y_residualTextualTrains, y_priceResidualTextualTrains, X_residualTextualValidations, y_residualTextualValidations, y_priceResidualTextualValidations, X_residualTextualTests, y_residualTextualTests, y_priceResidualTextualTests = [], [], [], [], [], [], [], [], []

# Defines lists for train, validation, test datasets for each component (price before)
y_priceTrendNumericalBeforeTrains, y_priceTrendNumericalBeforeValidations, y_priceTrendNumericalBeforeTests = [], [], []
y_priceTrendTextualBeforeTrains, y_priceTrendTextualBeforeValidations, y_priceTrendTextualBeforeTests = [], [], []
y_priceSeasonalBeforeTrains, y_priceSeasonalBeforeValidations, y_priceSeasonalBeforeTests = [], [], []
y_priceResidualNumericalBeforeTrains, y_priceResidualNumericalBeforeValidations, y_priceResidualNumericalBeforeTests = [], [], []
y_priceResidualTextualBeforeTrains, y_priceResidualTextualBeforeValidations, y_priceResidualTextualBeforeTests = [], [], []

# Defines the fold boundaries (3 folds cross-validations)
folds = [
    {"train": (0, 676), "validation": (676, 932), "test": (932, 1188)},
    {"train": (0, 932), "validation": (932, 1188), "test": (1188, 1444)},
    {"train": (0, 1188), "validation": (1188, 1444), "test": (1444, 1700)}
] 

# Defines a function to split data into three folds for cross-validations for train, validation, and test datasets
def splitFolds(X, y, y_price, y_priceBefore, folds):
    X_trains, y_trains, y_priceTrains, y_priceBeforeTrains = [], [], [], []
    X_validations, y_validations, y_priceValidations, y_priceBeforeValidations = [], [], [], []
    X_tests, y_tests, y_priceTests, y_priceBeforeTests = [], [], [], []

    for fold in folds:
        trainStart, trainEnd = fold["train"]
        valStart, valEnd = fold["validation"]
        testStart, testEnd = fold["test"]
        # Train
        X_trains.append(X[trainStart:trainEnd])
        y_trains.append(y[trainStart:trainEnd])
        y_priceTrains.append(y_price[trainStart:trainEnd])
        y_priceBeforeTrains.append(y_priceBefore[trainStart:trainEnd])
        # Validation
        X_validations.append(X[valStart:valEnd])
        y_validations.append(y[valStart:valEnd])
        y_priceValidations.append(y_price[valStart:valEnd])
        y_priceBeforeValidations.append(y_priceBefore[valStart:valEnd])
        # Test
        X_tests.append(X[testStart:testEnd])
        y_tests.append(y[testStart:testEnd])
        y_priceTests.append(y_price[testStart:testEnd])
        y_priceBeforeTests.append(y_priceBefore[testStart:testEnd])

    return (X_trains, y_trains, y_priceTrains,
            X_validations, y_validations, y_priceValidations,
            X_tests, y_tests, y_priceTests,
            y_priceBeforeTrains, y_priceBeforeValidations, y_priceBeforeTests)

# Calls the function for splitting the train, validation, and tests datasets into 3 folds
(X_trendNumericalTrains, y_trendNumericalTrains, y_priceTrendNumericalTrains,
 X_trendNumericalValidations, y_trendNumericalValidations, y_priceTrendNumericalValidations,
 X_trendNumericalTests, y_trendNumericalTests, y_priceTrendNumericalTests,
 y_priceTrendNumericalBeforeTrains, y_priceTrendNumericalBeforeValidations, y_priceTrendNumericalBeforeTests) = splitFolds(X_trendNumerical, y_trendNumerical, y_priceTrendNumerical, y_priceBeforeTrendNumerical, folds)

(X_trendTextualTrains, y_trendTextualTrains, y_priceTrendTextualTrains,
 X_trendTextualValidations, y_trendTextualValidations, y_priceTrendTextualValidations,
 X_trendTextualTests, y_trendTextualTests, y_priceTrendTextualTests,
 y_priceTrendTextualBeforeTrains, y_priceTrendTextualBeforeValidations, y_priceTrendTextualBeforeTests) = splitFolds(X_trendTextual, y_trendTextual, y_priceTrendTextual, y_priceBeforeTrendTextual, folds)

(X_seasonalTrains, y_seasonalTrains, y_priceSeasonalTrains,
 X_seasonalValidations, y_seasonalValidations, y_priceSeasonalValidations,
 X_seasonalTests, y_seasonalTests, y_priceSeasonalTests,
 y_priceSeasonalBeforeTrains, y_priceSeasonalBeforeValidations, y_priceSeasonalBeforeTests) = splitFolds(X_seasonal, y_seasonal, y_priceSeasonal, y_priceBeforeSeasonal, folds)

(X_residualNumericalTrains, y_residualNumericalTrains, y_priceResidualNumericalTrains,
 X_residualNumericalValidations, y_residualNumericalValidations, y_priceResidualNumericalValidations,
 X_residualNumericalTests, y_residualNumericalTests, y_priceResidualNumericalTests,
 y_priceResidualNumericalBeforeTrains, y_priceResidualNumericalBeforeValidations, y_priceResidualNumericalBeforeTests) = splitFolds(X_residualNumerical, y_residualNumerical, y_priceResidualNumerical, y_priceBeforeResidualNumerical, folds)

(X_residualTextualTrains, y_residualTextualTrains, y_priceResidualTextualTrains,
 X_residualTextualValidations, y_residualTextualValidations, y_priceResidualTextualValidations,
 X_residualTextualTests, y_residualTextualTests, y_priceResidualTextualTests,
 y_priceResidualTextualBeforeTrains, y_priceResidualTextualBeforeValidations, y_priceResidualTextualBeforeTests) = splitFolds(X_residualTextual, y_residualTextual, y_priceResidualTextual, y_priceBeforeResidualTextual, folds)

# # Defines a helper function to print the shapes of cross-validation splits
# def printFoldShapes(name, X_trains, y_trains, y_priceTrains, 
#                     X_validations, y_validations, y_priceValidations, 
#                     X_tests, y_tests, y_priceTests, 
#                     y_priceBeforeTrains, y_priceBeforeValidations, y_priceBeforeTests):

#     print(f"\n====== {name.upper()} COMPONENT ======")
#     for i in range(len(X_trains)):
#         print(f"Fold {i+1}")
#         print(f"  Train:       X={X_trains[i].shape}, y={y_trains[i].shape}, y_price={y_priceTrains[i].shape}, y_price_before={y_priceBeforeTrains[i].shape}")
#         print(f"  Validation:  X={X_validations[i].shape}, y={y_validations[i].shape}, y_price={y_priceValidations[i].shape}, y_price_before={y_priceBeforeValidations[i].shape}")
#         print(f"  Test:        X={X_tests[i].shape}, y={y_tests[i].shape}, y_price={y_priceTests[i].shape}, y_price_before={y_priceBeforeTests[i].shape}")
#     print("=====================================")

# # Calls the helper function to display shape summaries for all components
# printFoldShapes("Trend (Numerical)", 
#     X_trendNumericalTrains, y_trendNumericalTrains, y_priceTrendNumericalTrains, 
#     X_trendNumericalValidations, y_trendNumericalValidations, y_priceTrendNumericalValidations,
#     X_trendNumericalTests, y_trendNumericalTests, y_priceTrendNumericalTests,
#     y_priceTrendNumericalBeforeTrains, y_priceTrendNumericalBeforeValidations, y_priceTrendNumericalBeforeTests
# )

# printFoldShapes("Trend (Textual)", 
#     X_trendTextualTrains, y_trendTextualTrains, y_priceTrendTextualTrains, 
#     X_trendTextualValidations, y_trendTextualValidations, y_priceTrendTextualValidations,
#     X_trendTextualTests, y_trendTextualTests, y_priceTrendTextualTests,
#     y_priceTrendTextualBeforeTrains, y_priceTrendTextualBeforeValidations, y_priceTrendTextualBeforeTests
# )

# printFoldShapes("Seasonal", 
#     X_seasonalTrains, y_seasonalTrains, y_priceSeasonalTrains, 
#     X_seasonalValidations, y_seasonalValidations, y_priceSeasonalValidations,
#     X_seasonalTests, y_seasonalTests, y_priceSeasonalTests,
#     y_priceSeasonalBeforeTrains, y_priceSeasonalBeforeValidations, y_priceSeasonalBeforeTests
# )

# printFoldShapes("Residual (Numerical)", 
#     X_residualNumericalTrains, y_residualNumericalTrains, y_priceResidualNumericalTrains, 
#     X_residualNumericalValidations, y_residualNumericalValidations, y_priceResidualNumericalValidations,
#     X_residualNumericalTests, y_residualNumericalTests, y_priceResidualNumericalTests,
#     y_priceResidualNumericalBeforeTrains, y_priceResidualNumericalBeforeValidations, y_priceResidualNumericalBeforeTests
# )

# printFoldShapes("Residual (Textual)", 
#     X_residualTextualTrains, y_residualTextualTrains, y_priceResidualTextualTrains, 
#     X_residualTextualValidations, y_residualTextualValidations, y_priceResidualTextualValidations,
#     X_residualTextualTests, y_residualTextualTests, y_priceResidualTextualTests,
#     y_priceResidualTextualBeforeTrains, y_priceResidualTextualBeforeValidations, y_priceResidualTextualBeforeTests
# )

# Trend Numerical component
# Fold 01
# Shape [676, 8, 432] [676, 8] [676, 8] [676, 1] [256, 8, 432] [256, 8] [256, 8] [256, 1] [256, 8, 432] [256, 8] [256, 8] [256, 1]
# Fold 02
# Shape [932, 8, 432] [932, 8] [932, 8] [932, 1] [256, 8, 432] [256, 8] [256, 8] [256, 1] [256, 8, 432] [256, 8] [256, 8] [256, 1]
# Fold 03
# Shape [1188, 8, 432] [1188, 8] [1188, 8] [1188, 1] [256, 8, 432] [256, 8] [256, 8] [256, 1] [256, 8, 432] [256, 8] [256, 8] [256, 1]

# Trend Textual component
# Fold 01
# Shape [676, 8, 768] [676, 8] [676, 8] [676, 1] [256, 8, 768] [256, 8] [256, 8] [256, 1] [256, 8, 768] [256, 8] [256, 8] [256, 1]
# Fold 02
# Shape [932, 8, 768] [932, 8] [932, 8] [932, 1] [256, 8, 768] [256, 8] [256, 8] [256, 1] [256, 8, 768] [256, 8] [256, 8] [256, 1]
# Fold 03
# Shape [1188, 8, 768] [1188, 8] [1188, 8] [1188, 1] [256, 8, 768] [256, 8] [256, 8] [256, 1] [256, 8, 768] [256, 8] [256, 8] [256, 1]

# Seasonal and Residual Numerical components
# Fold 01
# Shape [676, 8, 2] [676, 8] [676, 8] [676, 1] [256, 8, 2] [256, 8] [256, 8] [256, 1] [256, 8, 2] [256, 8] [256, 8] [256, 1]
# Fold 02
# Shape [932, 8, 2] [932, 8] [932, 8] [932, 1] [256, 8, 2] [256, 8] [256, 8] [256, 1] [256, 8, 2] [256, 8] [256, 8] [256, 1]
# Fold 03
# Shape [1188, 8, 2] [1188, 8] [1188, 8] [1188, 1] [256, 8, 2] [256, 8] [256, 8] [256, 1] [256, 8, 2] [256, 8] [256, 8] [256, 1]

# Residual Textual component
# Fold 01
# Shape [676, 8, 768] [676, 8] [676, 8] [676, 1] [256, 8, 768] [256, 8] [256, 8] [256, 1] [256, 8, 768] [256, 8] [256, 8] [256, 1]
# Fold 02
# Shape [932, 8, 768] [932, 8] [932, 8] [932, 1] [256, 8, 768] [256, 8] [256, 8] [256, 1] [256, 8, 768] [256, 8] [256, 8] [256, 1]
# Fold 03
# Shape [1188, 8, 768] [1188, 8] [1188, 8] [1188, 1] [256, 8, 768] [256, 8] [256, 8] [256, 1] [256, 8, 768] [256, 8] [256, 8] [256, 1]

# Consolidates train, validation, and test datasets into dictionaries
X_trains = {'trendNumerical': X_trendNumericalTrains, 'trendTextual': X_trendTextualTrains, 'seasonal': X_seasonalTrains, 'residualNumerical': X_residualNumericalTrains, 'residualTextual': X_residualTextualTrains}
y_trains = {'trendNumerical': y_trendNumericalTrains, 'trendTextual': y_trendTextualTrains, 'seasonal': y_seasonalTrains, 'residualNumerical': y_residualNumericalTrains, 'residualTextual': y_residualTextualTrains}
y_priceTrains = {'trendNumerical': y_priceTrendNumericalTrains, 'trendTextual': y_priceTrendTextualTrains, 'seasonal': y_priceSeasonalTrains, 'residualNumerical': y_priceResidualNumericalTrains, 'residualTextual': y_priceResidualTextualTrains}
y_priceBeforeTrains = {'trendNumerical': y_priceTrendNumericalBeforeTrains, 'trendTextual': y_priceTrendTextualBeforeTrains, 'seasonal': y_priceSeasonalBeforeTrains, 'residualNumerical': y_priceResidualNumericalBeforeTrains, 'residualTextual': y_priceResidualTextualBeforeTrains}
X_validations = {'trendNumerical': X_trendNumericalValidations, 'trendTextual': X_trendTextualValidations, 'seasonal': X_seasonalValidations, 'residualNumerical': X_residualNumericalValidations, 'residualTextual': X_residualTextualValidations}
y_validations = {'trendNumerical': y_trendNumericalValidations, 'trendTextual': y_trendTextualValidations, 'seasonal': y_seasonalValidations, 'residualNumerical': y_residualNumericalValidations, 'residualTextual': y_residualTextualValidations}
y_priceValidations = {'trendNumerical': y_priceTrendNumericalValidations, 'trendTextual': y_priceTrendTextualValidations, 'seasonal': y_priceSeasonalValidations, 'residualNumerical': y_priceResidualNumericalValidations, 'residualTextual': y_priceResidualTextualValidations}
y_priceBeforeValidations = {'trendNumerical': y_priceTrendNumericalBeforeValidations, 'trendTextual': y_priceTrendTextualBeforeValidations, 'seasonal': y_priceSeasonalBeforeValidations, 'residualNumerical': y_priceResidualNumericalBeforeValidations, 'residualTextual': y_priceResidualTextualBeforeValidations}
X_tests = {'trendNumerical': X_trendNumericalTests, 'trendTextual': X_trendTextualTests, 'seasonal': X_seasonalTests, 'residualNumerical': X_residualNumericalTests, 'residualTextual': X_residualTextualTests}
y_tests = {'trendNumerical': y_trendNumericalTests, 'trendTextual': y_trendTextualTests, 'seasonal': y_seasonalTests, 'residualNumerical': y_residualNumericalTests, 'residualTextual': y_residualTextualTests}
y_priceTests = {'trendNumerical': y_priceTrendNumericalTests, 'trendTextual': y_priceTrendTextualTests, 'seasonal': y_priceSeasonalTests, 'residualNumerical': y_priceResidualNumericalTests, 'residualTextual': y_priceResidualTextualTests}
y_priceBeforeTests = {'trendNumerical': y_priceTrendNumericalBeforeTests, 'trendTextual': y_priceTrendTextualBeforeTests, 'seasonal': y_priceSeasonalBeforeTests, 'residualNumerical': y_priceResidualNumericalBeforeTests, 'residualTextual': y_priceResidualTextualBeforeTests}
# -----------------------------------------------------------------------------------------------------------------------------------

# -----------------------------------------------------------------------------------------------------------------------------------
#--- Builds the RNN Models for the trend numerical, trend textual, seasonal, and residuals numerical, and residual textual independently ---
# Defines a model outputing the number of outpuTimesteps on the final layer silmutaneously

def TrendNumericalModel(timesteps, features, outputTimesteps):
    model = Sequential()
    model.add(InputLayer(shape=(timesteps, features)))
    model.add(LSTM(512, return_sequences=False))
    model.add(Dense(outputTimesteps, activation='linear'))  
    model.compile(optimizer=Adam(), loss=customTrendLoss(lambda_=0.3, delta=1, penalty_factor=0.5, threshold=0.05))
    return model

def TrendTextualModel(timesteps, features, outputTimesteps):
    model = Sequential()
    model.add(InputLayer(shape=(timesteps, features)))
    model.add(LSTM(100, return_sequences=False))
    model.add(Dense(outputTimesteps, activation='linear'))  
    model.compile(optimizer=Adam(), loss=customTrendLoss(lambda_=0.2, delta=1, penalty_factor=0.5, threshold=0.05))
    return model

def SeasonalModel(timesteps, features, outputTimesteps):
    model = Sequential()
    model.add(InputLayer(shape=(timesteps, features)))
    model.add(LSTM(200, return_sequences=False))
    model.add(Dense(outputTimesteps, activation='linear'))  
    model.compile(optimizer=Adam(), loss='MSE')
    return model

def ResidualNumericalModel(timesteps, features, outputTimesteps):
    model = Sequential()
    model.add(InputLayer(shape=(timesteps, features)))
    model.add(LSTM(10, return_sequences=False))
    model.add(Dense(outputTimesteps, activation='linear'))  
    model.compile(optimizer=Adam(), loss=customTrendLoss(lambda_=0.2, delta=1.5, penalty_factor=0, threshold=0))
    return model

def ResidualTextualModel(timesteps, features, outputTimesteps):
    model = Sequential()
    model.add(InputLayer(shape=(timesteps, features)))
    model.add(LSTM(256, return_sequences=False))
    model.add(Dense(outputTimesteps, activation='linear'))  
    model.compile(optimizer=Adam(), loss=customTrendLoss(lambda_=0.2, delta=1.5, penalty_factor=0, threshold=0))
    return model

# Defines lists to store models, training losses, and validation losses for each components-folds combination
models = {'trendNumerical': [], 'trendTextual': [], 'seasonal': [], 'residualNumerical': [], 'residualTextual': []}
trainLosses = {'trendNumerical': [], 'trendTextual': [], 'seasonal': [], 'residualNumerical': [], 'residualTextual': []}
valLosses = {'trendNumerical': [], 'trendTextual': [], 'seasonal': [], 'residualNumerical': [], 'residualTextual': []}
# Defines a directory to save the best models for each components-folds combination
modelSaveDirectory = './BestMacroeconomicsOnly(LSTM)/'
os.makedirs(modelSaveDirectory, exist_ok=True)

# Stores best validation losses from each fold per component
bestValLossesTrend = {'trendNumerical': [], 'trendTextual': []}
bestValLossesResiduals = {'residualNumerical': [], 'residualTextual': []}

class LiveLossPlotCallback(tf.keras.callbacks.Callback):
    def __init__(self):
        self.train_losses = []
        self.val_losses = []

    def on_epoch_end(self, epoch, logs=None):
        self.train_losses.append(logs.get('loss'))
        self.val_losses.append(logs.get('val_loss'))
        plt.clf()
        plt.plot(self.train_losses, label='Training Loss')
        plt.plot(self.val_losses, label='Validation Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss (MSE)')
        plt.title('Live Training vs Validation Loss')
        plt.legend()
        plt.grid(True)
        display.clear_output(wait=True)
        display.display(plt.gcf())

# Defines a function to train and save the best models for each components-folds combination
def trainSaveBestModel(X_train, y_train, X_validation, y_validation, epochs, batch_size, featureType, idx):
    if featureType == 'trendNumerical':
        regressor = TrendNumericalModel(timesteps, X_train.shape[-1], outputTimesteps)
    if featureType == 'trendTextual':
        regressor = TrendTextualModel(timesteps, X_train.shape[-1], outputTimesteps)
    elif featureType == 'seasonal':
        regressor = SeasonalModel(timesteps, X_train.shape[-1], outputTimesteps)
    elif featureType == 'residualNumerical':
        regressor = ResidualNumericalModel(timesteps, X_train.shape[-1], outputTimesteps)
    elif featureType == 'residualTextual':
        regressor = ResidualTextualModel(timesteps, X_train.shape[-1], outputTimesteps)

    earlyStopping = EarlyStopping(
        monitor='val_loss', 
        patience=30,
        restore_best_weights=True,
        verbose=1
    )

    history = regressor.fit(
        X_train, y_train,
        validation_data=(X_validation, y_validation),
        epochs=epochs,
        batch_size=batch_size,
        shuffle=False,
        callbacks=[earlyStopping],
        verbose=1
    )

    bestValLoss = min(history.history['val_loss'])
    print(f"Best Validation Loss for {featureType.capitalize()} Fold {idx + 1}: {bestValLoss:.3f}")
    if featureType == 'trendNumerical' or featureType == 'trendTextual':
        bestValLossesTrend[featureType].append(bestValLoss)
    if featureType == 'residualNumerical' or featureType == 'residualTextual':
        bestValLossesResiduals[featureType].append(bestValLoss)
    modelSavePath = os.path.join(modelSaveDirectory, f'bestModel_{featureType}_fold0{idx + 1}.weights.h5')
    regressor.save_weights(modelSavePath)
    print(f"Saved best model weights to {modelSavePath}")

    return regressor, history.history

# Calls the train and save the best models functions for each components-folds combination
for featureType in ['trendNumerical', 'trendTextual', 'seasonal', 'residualNumerical', 'residualTextual']:
    for idx, (X_train, y_train, X_val, y_val) in enumerate(zip(X_trains[featureType], y_trains[featureType], X_validations[featureType], y_validations[featureType])):
        print(f"\nTraining and saving the best model for {featureType.capitalize()} fold {idx + 1}...")
        if featureType == 'trendNumerical':
            model, history = trainSaveBestModel(X_train, y_train, X_val, y_val, epochs=1000, batch_size=128, featureType=featureType, idx=idx)
        elif featureType == 'trendTextual':
            model, history = trainSaveBestModel(X_train, y_train, X_val, y_val, epochs=1, batch_size=128, featureType=featureType, idx=idx)
        elif featureType == 'seasonal':
            model, history = trainSaveBestModel(X_train, y_train, X_val, y_val, epochs=1, batch_size=64, featureType=featureType, idx=idx)
        elif featureType == 'residualNumerical':
            model, history = trainSaveBestModel(X_train, y_train, X_val, y_val, epochs=1000, batch_size=32, featureType=featureType, idx=idx)
        elif featureType == 'residualTextual':
            model, history = trainSaveBestModel(X_train, y_train, X_val, y_val, epochs=1, batch_size=32, featureType=featureType, idx=idx)
        models[featureType].append(model)
        trainLosses[featureType].append(history['loss'])
        valLosses[featureType].append(history['val_loss'])

# Visualizes the train and validation losses separately for each component
for featureType in ['trendNumerical', 'trendTextual', 'seasonal', 'residualNumerical', 'residualTextual']:
    plt.figure(figsize=(10, 5))
    for idx, (trainLoss, valLoss) in enumerate(zip(trainLosses[featureType], valLosses[featureType])):
        plt.plot(range(len(trainLoss)), trainLoss, label=f'Train - Fold {idx + 1}')
        plt.plot(range(len(valLoss)), valLoss, linestyle='dashed', label=f'Validation - Fold {idx + 1}')
    plt.title(f'{featureType.capitalize()} Component: Training and Validation Loss per Epoch', fontsize=12)
    plt.xlabel('Epoch', fontsize=10)
    plt.ylabel('Loss (MSE)', fontsize=10)
    plt.legend(prop={'size': 8})
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# Defines a function to compute fold-specific weights using inverse-MSE
def computeFoldWeights(bestLossesDict):
    numFolds = len(next(iter(bestLossesDict.values())))
    weights_per_fold = []
    for fold_idx in range(numFolds):
        inv_mse = {k: 1 / bestLossesDict[k][fold_idx] for k in bestLossesDict}
        total_inv = sum(inv_mse.values())
        weights = {k: round(v / total_inv, 2) for k, v in inv_mse.items()}
        weights_per_fold.append(weights)
    return weights_per_fold

# Computes weights
weightsTrend = computeFoldWeights(bestValLossesTrend)
weightsResiduals = computeFoldWeights(bestValLossesResiduals)

# print(bestValLossesTrend)
# print(bestValLossesResiduals)

# Prints weights for trend component
print("=== Fold-Specific Weights for Trend Component ===")
for i, fold_weights in enumerate(weightsTrend):
    print(f"Fold {i+1}:")
    for model, weight in fold_weights.items():
        print(f"  {model}: {weight}")
    print()

# Prints weights for residuals component
print("=== Fold-Specific Weights for Residual Component ===")
for i, fold_weights in enumerate(weightsResiduals):
    print(f"Fold {i+1}:")
    for model, weight in fold_weights.items():
        print(f"  {model}: {weight}")
    print()
# -----------------------------------------------------------------------------------------------------------------------------------

# -----------------------------------------------------------------------------------------------------------------------------------
#--- Defines function to check the prediction results by outputting the csv files ---

# Defines functions to save the actual and the predicted target variable values for each folds-components combination
# There will be 120 files outputed from this function
def savePredictionsToCSV(featureType, fold, currentStep, y_priceBeforeTrain_inverse, y_trainPrediction_inverse, y_priceTrainPrediction_inverse, y_train_inverse, y_priceTrain_inverse, label):
    dfPrediction = pd.DataFrame({
        'Price Before': np.round(y_priceBeforeTrain_inverse.flatten(), 2),
        'Predicted Price Difference': np.round(y_trainPrediction_inverse.flatten(), 2),
        'Predicted Price': np.round(y_priceTrainPrediction_inverse.flatten(), 2),
        'Actual Price Difference': np.round(y_train_inverse.flatten(), 2),
        'Actual Price': np.round(y_priceTrain_inverse.flatten(), 2),
        
    })
    csvOutputPath = f'./{label}_fold0{fold + 1}_{featureType}(0{currentStep + 1}).csv'
    dfPrediction.to_csv(csvOutputPath, index=False)

# Defines functions to save the combined actual and the predicted target variable values for each folds combination
# There will be 24 files outputed from this function
def saveOverallPredictionsToCSV(fold, currentStep, y_combinedPriceTrain_inverse, y_combinedPriceTrainPrediction_inverse, label):
    # Creates a folder to save the results 
    outputDir = './BestMacroeconomicsOnlyPredictionResults(LSTM)'
    os.makedirs(outputDir, exist_ok=True)
    dfCombined = pd.DataFrame({
        'Price': np.round(y_combinedPriceTrain_inverse.flatten(), 2),
        'Predicted Price': np.round(y_combinedPriceTrainPrediction_inverse.flatten(), 2),
    })
    csvOutputPath = os.path.join(outputDir, f'{label}_fold0{int(fold[-2:])}_overall(0{int(currentStep) + 1}).csv')
    dfCombined.to_csv(csvOutputPath, index=False)
# -----------------------------------------------------------------------------------------------------------------------------------

# -----------------------------------------------------------------------------------------------------------------------------------
#--- Defines a function to test the test datasets (each components-folds combination)  ---

def TestModel(models, X_tests, y_tests, y_priceTests, y_priceBeforeTests, trendScaler, trend_diffScaler, seasonalScaler, seasonal_diffScaler, residualScaler, residual_diffScaler, outputTimesteps):
    
    # Puts the defined scalers on dictionaries according to each feature type (trend, seasonal, and residual)
    scalers = {
        'trendNumerical': (trendScaler, trend_diffScaler),
        'trendTextual': (trendScaler, trend_diffScaler),
        'seasonal': (seasonalScaler, seasonal_diffScaler),
        'residualNumerical': (residualScaler, residual_diffScaler),
        'residualTextual': (residualScaler, residual_diffScaler)
    }

    # Initializes metrics dictionaries for each timesteps-folds-metrics-components combination
    initialMetrics = {
        'mae': {'fold01': [], 'fold02': [], 'fold03': []}, 
        'mse': {'fold01': [], 'fold02': [], 'fold03': []}, 
        'rmse': {'fold01': [], 'fold02': [], 'fold03': []}, 
        'mape': {'fold01': [], 'fold02': [], 'fold03': []},
        'smape': {'fold01': [], 'fold02': [], 'fold03': []}
    }
    testMetrics = {
        'trendNumerical': copy.deepcopy(initialMetrics),
        'trendTextual': copy.deepcopy(initialMetrics),
        'seasonal': copy.deepcopy(initialMetrics),
        'residualNumerical': copy.deepcopy(initialMetrics),
        'residualTextual': copy.deepcopy(initialMetrics)
    }
    priceTestMetrics = copy.deepcopy(testMetrics)

    # Initializes metrics dictionaries for each timesteps-metrics-folds (additive methodology) (trend + seasonal + residuals prices)
    combinedPriceTestMetrics = {'mae': {}, 'mse': {}, 'rmse': {}, 'mape': {}, 'smape': {}, 'trendAcc01': {}, 'trendAcc02': {}, 'trendAcc03': {}}

    # Initializes dictionaries for each fold combination (additive methodology) (trend + seasonal + residuals prices)
    y_combinedPriceTests_inverse = {'fold01': {}, 'fold02': {}, 'fold03': {}}
    y_combinedPriceTestPredictions_inverse = {'fold01': {}, 'fold02': {}, 'fold03': {}}
    y_combinedPriceBeforeTests_inverse = {'fold01': {}, 'fold02': {}, 'fold03': {}}

    # For fusion of trend and residuals components (numerical and textual)
    trendNumericalPredsTests= {'fold01': {}, 'fold02': {}, 'fold03': {}}
    trendTextualPredsTests = {'fold01': {}, 'fold02': {}, 'fold03': {}}
    trendNumericalActualsTests = {'fold01': {}, 'fold02': {}, 'fold03': {}}
    trendNumericalActualBeforeTests = {'fold01': {}, 'fold02': {}, 'fold03': {}}

    residualNumericalPredsTests= {'fold01': {}, 'fold02': {}, 'fold03': {}}
    residualTextualPredsTests = {'fold01': {}, 'fold02': {}, 'fold03': {}}
    residualNumericalActualsTests = {'fold01': {}, 'fold02': {}, 'fold03': {}}
    residualNumericalActualBeforeTests = {'fold01': {}, 'fold02': {}, 'fold03': {}}

    # Loops through all the components
    for featureType in ['trendNumerical', 'trendTextual', 'seasonal', 'residualNumerical', 'residualTextual']:
        priceScaler, price_diffScaler = scalers[featureType]
        print(f"\nTesting on {featureType.capitalize()} Models...")

        # Loops through all the folds
        for idx, (X_test, y_test, y_priceTest, y_priceBeforeTest, model) in enumerate(zip(
            X_tests[featureType], y_tests[featureType], y_priceTests[featureType], y_priceBeforeTests[featureType], models[featureType])):
        
            foldKey = f'fold0{idx + 1}'
            print(f"\nTesting on Fold 0{idx + 1}...")

            # Predicts all future timesteps at once by calling the trained model
            y_testPrediction = model.predict(X_test)

            # Because in the trend component this program predicts the price difference
            if featureType in ['trendNumerical', 'trendTextual']:
                # Inverses transform predictions and actual prices difference
                y_testPrediction_inverse = np.round(price_diffScaler.inverse_transform(y_testPrediction), 2)
                y_test_inverse = np.round(price_diffScaler.inverse_transform(y_test), 2)
                # Restores and inverses transform actual price before values
                y_priceBeforeTest_inverse = priceScaler.inverse_transform(y_priceBeforeTest.reshape(-1, 1))
                # Computes cumulative sum across timesteps starting from retrieves price before values
                y_priceTestPrediction_inverse = np.cumsum(np.hstack([y_priceBeforeTest_inverse, y_testPrediction_inverse]), axis=1)[:, 1:]
                y_priceTest_inverse = np.round(priceScaler.inverse_transform(y_priceTest), 2) 

                # Saves for fusion
                for t in range(outputTimesteps):
                    if featureType == 'trendNumerical':
                        trendNumericalPredsTests[foldKey][t] = y_priceTestPrediction_inverse[:, t]
                        trendNumericalActualsTests[foldKey][t] = y_priceTest_inverse[:, t]
                        if t == 0:
                            trendNumericalActualBeforeTests[foldKey][t] = y_priceBeforeTest_inverse.flatten()
                    elif featureType == 'trendTextual':
                        trendTextualPredsTests[foldKey][t] = y_priceTestPrediction_inverse[:, t]

            # Because in the seasonal and residual components this program directly predicts the price
            elif featureType in ['seasonal', 'residualNumerical', 'residualTextual']:
                # Note: the y_train_inverse and the y_priceTrain_inverse will have the same values
                # Inverses transform predictions and actual prices
                y_testPrediction_inverse = np.round(priceScaler.inverse_transform(y_testPrediction), 2)
                y_test_inverse = np.round(priceScaler.inverse_transform(y_test), 2)
                # Restores and inverses transform actual price before values
                y_priceBeforeTest_inverse = priceScaler.inverse_transform(y_priceBeforeTest.reshape(-1, 1))
                # Because we predicts the price, we just put the prediction into the price prediction variable
                y_priceTestPrediction_inverse = np.round(priceScaler.inverse_transform(y_testPrediction), 2)
                y_priceTest_inverse = np.round(priceScaler.inverse_transform(y_priceTest), 2)

                if featureType in ['seasonal']:
                    y_priceTestPrediction_inverse = np.round(priceScaler.inverse_transform(y_priceTest), 2)

                if featureType == 'seasonal': 
                    for t in range (outputTimesteps):
                        if t not in y_combinedPriceTests_inverse[foldKey]:
                            y_combinedPriceTests_inverse[foldKey][t] = y_priceTest_inverse[:,t]
                            y_combinedPriceTestPredictions_inverse[foldKey][t] = y_priceTestPrediction_inverse[:,t]
                        else:
                            y_combinedPriceTests_inverse[foldKey][t] += y_priceTest_inverse[:,t]
                            y_combinedPriceTestPredictions_inverse[foldKey][t] += y_priceTestPrediction_inverse[:,t]
                    
                        # Gets the price before at t = 0 for counting the trend accuracy
                        if t == 0:
                            if t not in y_combinedPriceBeforeTests_inverse[foldKey]:
                                y_combinedPriceBeforeTests_inverse[foldKey][t] = y_priceBeforeTest_inverse.flatten()
                            else:
                                y_combinedPriceBeforeTests_inverse[foldKey][t] += y_priceBeforeTest_inverse.flatten()
                
                if featureType in ['residualNumerical', 'residualTextual']:
                    for t in range(outputTimesteps):
                        if featureType == 'residualNumerical':
                            residualNumericalPredsTests[foldKey][t] = y_priceTestPrediction_inverse[:, t]
                            residualNumericalActualsTests[foldKey][t] = y_priceTest_inverse[:, t]
                            if t == 0:
                                residualNumericalActualBeforeTests[foldKey][t] = y_priceBeforeTest_inverse.flatten()
                        elif featureType == 'residualTextual':
                            residualTextualPredsTests[foldKey][t] = y_priceTestPrediction_inverse[:, t]

            # Computes and stores the metrices for predicted price differences and predicted prices
            for t in range(outputTimesteps):
                testMetrics[featureType]['mae'][foldKey].append(mean_absolute_error(y_test_inverse[:, t], y_testPrediction_inverse[:, t]))
                testMetrics[featureType]['mse'][foldKey].append(mean_squared_error(y_test_inverse[:, t], y_testPrediction_inverse[:, t]))
                testMetrics[featureType]['rmse'][foldKey].append(np.sqrt(mean_squared_error(y_test_inverse[:, t], y_testPrediction_inverse[:, t])))
                testMetrics[featureType]['mape'][foldKey].append(MAPE(y_test_inverse[:, t], y_testPrediction_inverse[:, t]))
                testMetrics[featureType]['smape'][foldKey].append(SMAPE(y_test_inverse[:, t], y_testPrediction_inverse[:, t]))

                priceTestMetrics[featureType]['mae'][foldKey].append(mean_absolute_error(y_priceTest_inverse[:, t], y_priceTestPrediction_inverse[:, t]))
                priceTestMetrics[featureType]['mse'][foldKey].append(mean_squared_error(y_priceTest_inverse[:, t], y_priceTestPrediction_inverse[:, t]))
                priceTestMetrics[featureType]['rmse'][foldKey].append(np.sqrt(mean_squared_error(y_priceTest_inverse[:, t], y_priceTestPrediction_inverse[:, t])))
                priceTestMetrics[featureType]['mape'][foldKey].append(MAPE(y_priceTest_inverse[:, t], y_priceTestPrediction_inverse[:, t]))  
                priceTestMetrics[featureType]['smape'][foldKey].append(SMAPE(y_priceTest_inverse[:, t], y_priceTestPrediction_inverse[:, t]))  

            # # Saves the prediction to a .CSV file for all timesteps
            # # Should be outputed 120 .csv files
            # for t in range(outputTimesteps):
            #     savePredictionsToCSV(
            #         featureType, idx, t, 
            #         y_priceBeforeTest_inverse,
            #         y_testPrediction_inverse[:, t].reshape(-1, 1), 
            #         y_priceTestPrediction_inverse[:, t].reshape(-1, 1), 
            #         y_test_inverse[:, t].reshape(-1, 1), 
            #         y_priceTest_inverse[:, t].reshape(-1, 1), 
            #         'test'
            #     )

        # # Iterates over timesteps to print the metrics of the components-folds combination
        # for fold_num in range(len(folds)):
        #     foldKey = f'fold0{fold_num + 1}'
        #     for t in range(outputTimesteps):
        #         print(f"\n(Test) Price Metrics for {featureType.capitalize()} {foldKey.capitalize()}, Timestep {t + 1}: "
        #                 f"MAE: {priceTestMetrics[featureType]['mae'][foldKey][t]:.3f}, "
        #                 f"MSE: {priceTestMetrics[featureType]['mse'][foldKey][t]:.3f}, "
        #                 f"RMSE: {priceTestMetrics[featureType]['rmse'][foldKey][t]:.3f}, "
        #                 f"MAPE: {priceTestMetrics[featureType]['mape'][foldKey][t]:.3f}, "
        #                 f"SMAPE: {priceTestMetrics[featureType]['smape'][foldKey][t]:.3f}")
        #     print('\n') 

    # Applies late fusion to trend components in test
    print("\nLate fusion of trend components (Test) (Trend)...")
    for fold_num in range(len(folds)):
        foldKey = f'fold0{fold_num + 1}'
        for t in range(outputTimesteps):
            weight_num = weightsTrend[fold_num]['trendNumerical']
            weight_text = weightsTrend[fold_num]['trendTextual']
                
            pred_num = trendNumericalPredsTests[foldKey][t]
            pred_text = trendTextualPredsTests[foldKey][t]
            actual = trendNumericalActualsTests[foldKey][t]
            before = trendNumericalActualBeforeTests[foldKey][0]

            fused_pred = (pred_num * (1)) + (pred_text * (0))
            
            # Update combined predictions
            y_combinedPriceTestPredictions_inverse[foldKey][t] += fused_pred
            y_combinedPriceTests_inverse[foldKey][t] += actual

            # Save actual price before for trend accuracy calculation
            if t == 0:
                y_combinedPriceBeforeTests_inverse[foldKey][t] += before
    
    print("\nLate fusion of trend components (Test) (Residuals)...")
    for fold_num in range(len(folds)):
        foldKey = f'fold0{fold_num + 1}'
        for t in range(outputTimesteps):
            weight_num = weightsResiduals[fold_num]['residualNumerical']
            weight_text = weightsResiduals[fold_num]['residualTextual']
                
            pred_num = residualNumericalPredsTests[foldKey][t]
            pred_text = residualTextualPredsTests[foldKey][t]
            actual = residualNumericalActualsTests[foldKey][t]
            before = residualNumericalActualBeforeTests[foldKey][0]

            fused_pred = (pred_num * (1)) + (pred_text * (0))
            
            # Update combined predictions
            y_combinedPriceTestPredictions_inverse[foldKey][t] += fused_pred
            y_combinedPriceTests_inverse[foldKey][t] += actual

            # Save actual price before for trend accuracy calculation
            if t == 0:
                y_combinedPriceBeforeTests_inverse[foldKey][t] += before

    # Saves and output a .csv file for the overall actual and predicted prices for each fold combination (additive methodology)
    # Should be outputed 24 .csv files
    for foldKey in ['fold01', 'fold02', 'fold03']:
        for t in y_combinedPriceTests_inverse[foldKey].keys():
            saveOverallPredictionsToCSV(foldKey, t, y_combinedPriceTests_inverse[foldKey][t], y_combinedPriceTestPredictions_inverse[foldKey][t], 'test')

    # Calculates the metrics of the overall components for each timesteps-folds combination
    # Loops over folds
    for foldKey in ['fold01', 'fold02', 'fold03']:
        # Loops over timesteps
        for t in range(outputTimesteps):
            if t in y_combinedPriceTests_inverse[foldKey]:
                combinedPriceTestMetricsMAE = mean_absolute_error(y_combinedPriceTests_inverse[foldKey][t], y_combinedPriceTestPredictions_inverse[foldKey][t])
                combinedPriceTestMetricsMSE = mean_squared_error(y_combinedPriceTests_inverse[foldKey][t], y_combinedPriceTestPredictions_inverse[foldKey][t])
                combinedPriceTestMetricsRMSE = np.sqrt(combinedPriceTestMetricsMSE)
                combinedPriceTestMetricsMAPE = MAPE(y_combinedPriceTests_inverse[foldKey][t], y_combinedPriceTestPredictions_inverse[foldKey][t])
                combinedPriceTestMetricsSMAPE = SMAPE(y_combinedPriceTests_inverse[foldKey][t], y_combinedPriceTestPredictions_inverse[foldKey][t])
                combinedPriceTestMetricsTrendAcc01 = trendAccuracy(
                    y_actualBefore=y_combinedPriceBeforeTests_inverse[foldKey][0], 
                    y_actual=y_combinedPriceTests_inverse[foldKey][t], 
                    y_predictedBefore=y_combinedPriceBeforeTests_inverse[foldKey][0],  
                    y_predicted=y_combinedPriceTestPredictions_inverse[foldKey][t], threshold=1)
                combinedPriceTestMetricsTrendAcc02 = trendAccuracy(
                    y_actualBefore=y_combinedPriceBeforeTests_inverse[foldKey][0], 
                    y_actual=y_combinedPriceTests_inverse[foldKey][t], 
                    y_predictedBefore=y_combinedPriceBeforeTests_inverse[foldKey][0],  
                    y_predicted=y_combinedPriceTestPredictions_inverse[foldKey][t], threshold=3)
                combinedPriceTestMetricsTrendAcc03 = trendAccuracy(
                    y_actualBefore=y_combinedPriceBeforeTests_inverse[foldKey][0], 
                    y_actual=y_combinedPriceTests_inverse[foldKey][t], 
                    y_predictedBefore=y_combinedPriceBeforeTests_inverse[foldKey][0],  
                    y_predicted=y_combinedPriceTestPredictions_inverse[foldKey][t], threshold=8.75)

                for metric, value in zip(['mae', 'mse', 'rmse', 'mape', 'smape', 'trendAcc01', 'trendAcc02', 'trendAcc03'],
                    [combinedPriceTestMetricsMAE, combinedPriceTestMetricsMSE, combinedPriceTestMetricsRMSE, combinedPriceTestMetricsMAPE, combinedPriceTestMetricsSMAPE, combinedPriceTestMetricsTrendAcc01,
                        combinedPriceTestMetricsTrendAcc02,combinedPriceTestMetricsTrendAcc03]):
                    if t not in combinedPriceTestMetrics[metric]:
                        combinedPriceTestMetrics[metric][t] = []
                    combinedPriceTestMetrics[metric][t].append(value)

            # Outputs metrics for each timesteps-folds combination
            print(f"\n===== {foldKey.upper()} | Timestep {t+1} Results =====")
            print(f"MAE     : {combinedPriceTestMetricsMAE:.4f}")
            print(f"MSE     : {combinedPriceTestMetricsMSE:.4f}")
            print(f"RMSE    : {combinedPriceTestMetricsRMSE:.4f}")
            print(f"MAPE    : {combinedPriceTestMetricsMAPE:.4f}")
            print(f"SMAPE   : {combinedPriceTestMetricsSMAPE:.4f}")
            print(f"TrendAcc @ ±1%   : {combinedPriceTestMetricsTrendAcc01:.4f}")
            print(f"TrendAcc @ ±3%   : {combinedPriceTestMetricsTrendAcc02:.4f}")
            print(f"TrendAcc @ ±8.75%: {combinedPriceTestMetricsTrendAcc03:.4f}")
            print("="*45)

    # Calculates and outputs Mean and Standard Deviation of each Metric for each timesteps (Test Phase)
    print(f"\n=== Overall Price Metrics Summary (TEST) ===")
    for t in range(outputTimesteps):
        print(f"=== Timestep {t+1} ===")
        for metric in ['mae', 'mse', 'rmse', 'mape', 'smape', 'trendAcc01', 'trendAcc02', 'trendAcc03']:
            values = combinedPriceTestMetrics[metric].get(t, [])
            if values:
                mean_val = np.mean(values)
                std_val = np.std(values)
                print(f"{metric.upper():<10} - Mean: {mean_val:.4f}, Std: {std_val:.4f}")
        print("\n")

    # Generate and save final metric summary inside the function
    mean_table = {"Timestep": list(range(1, outputTimesteps + 1))}
    std_table = {"Timestep": list(range(1, outputTimesteps + 1))}
    metric_list = ['mae', 'mse', 'rmse', 'mape', 'smape', 'trendAcc01', 'trendAcc02', 'trendAcc03']

    for metric in metric_list:
        mean_vals = []
        std_vals = []
        for t in range(outputTimesteps):
            values = combinedPriceTestMetrics[metric].get(t, [])
            mean_vals.append(np.round(np.mean(values), 4))
            std_vals.append(np.round(np.std(values), 4))
        
        mean_table[metric.upper()] = mean_vals
        std_table[metric.upper()] = std_vals

    df_mean = pd.DataFrame(mean_table)
    df_std = pd.DataFrame(std_table)

    empty_row = pd.DataFrame([[""] + [""] * (len(df_mean.columns) - 1)], columns=df_mean.columns)
    final_df = pd.concat([df_mean, empty_row, df_std], ignore_index=True)
    final_df.to_csv("./BestMacroeconomicsOnly(LSTM).csv", index=False)
  
    # Stacks all timestep arrays horizontally (axis=1) to form new shape for all folds
    fold01_pred = np.column_stack([y_combinedPriceTestPredictions_inverse['fold01'][t] for t in range(outputTimesteps)])
    fold01_actual = np.column_stack([y_combinedPriceTests_inverse['fold01'][t] for t in range(outputTimesteps)])
    fold01_price_before = y_combinedPriceBeforeTests_inverse['fold01'][0]

    fold02_pred = np.column_stack([y_combinedPriceTestPredictions_inverse['fold02'][t] for t in range(outputTimesteps)])
    fold02_actual = np.column_stack([y_combinedPriceTests_inverse['fold02'][t] for t in range(outputTimesteps)])
    fold02_price_before = y_combinedPriceBeforeTests_inverse['fold02'][0]

    fold03_pred = np.column_stack([y_combinedPriceTestPredictions_inverse['fold03'][t] for t in range(outputTimesteps)])
    fold03_actual = np.column_stack([y_combinedPriceTests_inverse['fold03'][t] for t in range(outputTimesteps)])
    fold03_price_before = y_combinedPriceBeforeTests_inverse['fold03'][0]

    return (fold01_pred, fold01_actual, fold01_price_before,
            fold02_pred, fold02_actual, fold02_price_before,
            fold03_pred, fold03_actual, fold03_price_before)

# Calls the function to test datasets
(fold1_pred, fold1_actual, fold1_price_before,
 fold2_pred, fold2_actual, fold2_price_before,
 fold3_pred, fold3_actual, fold3_price_before) = TestModel(
    models=models,
    X_tests=X_tests,
    y_tests=y_tests,
    y_priceTests=y_priceTests,
    y_priceBeforeTests=y_priceBeforeTests,
    trendScaler=trendScaler,
    trend_diffScaler=trend_diffScaler,
    seasonalScaler=seasonalScaler,
    seasonal_diffScaler=seasonal_diffScaler,
    residualScaler=residualScaler,
    residual_diffScaler=residual_diffScaler,
    outputTimesteps=outputTimesteps
)

# print(fold3_pred.shape) # Shape [256, 8]
# print(fold3_actual.shape) # Shape [256, 8]
# print(fold3_price_before.shape) # Shape [256]

def visualize_fold_predictions(fold_pred, fold_actual, fold_price_before, fold_number):
    reshaped_preds = fold_price_before.reshape(-1, 1)
    for t in range(fold_pred.shape[1]):
        predictions = fold_pred[:, t]
        reshaped_preds = np.hstack([reshaped_preds, predictions.reshape(-1, 1)])

    first_actual = fold_price_before[:7]
    second_actual = fold_actual[:-7, 0]
    third_actual = fold_actual[-7:, :].T.flatten(order='C')
    combined_actual = np.concatenate([first_actual, second_actual, third_actual])

    plt.figure(figsize=(16, 8))
    plt.plot(
        combined_actual, 
        label='Actual Prices', 
        color='magenta', 
        linewidth=2, 
        alpha=0.9, 
        marker='o', 
        markersize=4
    )

    for row in range(len(reshaped_preds)):
        y_values = reshaped_preds[row]
        x_values = list(range(row, row + len(y_values) * interval, interval))[:len(y_values)]
        plt.plot(
            x_values, 
            y_values, 
            color=np.random.rand(3,), 
            linestyle='-', 
            linewidth=1.5, 
            alpha=0.8
        )

    plt.title(f'Oil Price Predict (Test) - Fold {fold_number}', fontsize=16)
    plt.xlabel('Time Period', fontsize=12)
    plt.ylabel('Price', fontsize=12)
    plt.grid(alpha=0.3)
    plt.legend(['Actual Prices', 'Predicted Sequences'], loc='upper right', fontsize=10)
    plt.tight_layout()
    plt.show()

# Calls the visualization function
visualize_fold_predictions(fold1_pred, fold1_actual, fold1_price_before, fold_number=1)
visualize_fold_predictions(fold2_pred, fold2_actual, fold2_price_before, fold_number=2)
visualize_fold_predictions(fold3_pred, fold3_actual, fold3_price_before, fold_number=3)